```sql
Goal:
- add a first set of history features to avoid leakage
- compare against the same train - validation split

Rule:
- history features must use only past records and should not change if re-fetched at different intervals
```

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np

from src.modeling.model_utils import model_results
from src.features.feature_utils import add_history_feature

In [2]:
from sklearn.metrics import roc_auc_score, average_precision_score
from lightgbm import LGBMClassifier

#### 1. Load the datasets

In [3]:
train_df=pd.read_pickle('../data/interim/train_joined_split.pkl')

In [4]:
valid_df=pd.read_pickle('../data/interim/valid_joined_split.pkl')

In [5]:
train_df.shape, valid_df.shape

((472432, 434), (118108, 434))

In [6]:
train_df = train_df.sort_values("TransactionDT").reset_index(drop=True)
valid_df = valid_df.sort_values("TransactionDT").reset_index(drop=True)

In [7]:
base_features=["TransactionAmt",
               "ProductCD",
               "card1", "card2", "card3", "card4", "card5", "card6",
               "addr1", "addr2",
               "P_emaildomain", "R_emaildomain",
               "DeviceType", "DeviceInfo"
              ]
target_col="isFraud"

In [8]:
available_features = [c for c in base_features if c in train_df.columns]

In [9]:
# Adding some transformation on features
for df_ in [train_df,valid_df]:
    df_["log_txnamt"]=np.log1p(df_["TransactionAmt"]) #log1p works best for skewed and zero values well
    df_["has_identity"]=(df_[["DeviceType", "DeviceInfo"]].notna().any(axis=1).astype(int)) #Any one value is available

In [10]:
new_features = ["log_txnamt", "has_identity"]

In [11]:
feature_cols=available_features+new_features

#### Model Training and Result Function

In [12]:
# model_input_data is now imported from src.modeling.model_utils!

In [13]:
# model_results is now imported from src.modeling.model_utils!

#### Baseline Results

In [14]:
baseline_result = model_results(train_df, valid_df, feature_cols)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Number of positive: 16599, number of negative: 455833
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.027217 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1808
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 16
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


In [15]:
print("Baseline ROC-AUC:", round(baseline_result["roc_auc"], 5))
print("Baseline PR-AUC :", round(baseline_result["pr_auc"], 5))

Baseline ROC-AUC: 0.80776
Baseline PR-AUC : 0.23062


#### Adding first history feature

```sql
Approach:

For each entity (e.g. card/user), build history-based features using only past transactions.

Base preparation:
- Work on a copy of dataset
- Sort data by time_col (chronological order)

For each entity:
    Initialize:
        past_amounts = []
        last_time = None

    For each transaction (in time order):

        Transaction count:
            count = len(past_amounts)

        Average transaction amount:
            if past_amounts exists:
                avg = sum(past_amounts) / len(past_amounts)
            else:
                avg = NULL

        Time since last transaction:
            if last_time exists:
                time_diff = current_time - last_time
            else:
                time_diff = NULL

        Store computed values

        Update history:
            append current amount to past_amounts
            last_time = current_time

Output:
- txn count (historical)
- avg txn amount (historical)
- time since last txn

Key idea:
Only use past data (no future leakage)

```

In [16]:
# add_history_feature is now imported from src.features.feature_utils!

In [17]:
train_h = add_history_feature(train_df, "card1", prefix="card1")
valid_h = add_history_feature(valid_df, "card1", prefix="card1")

In [18]:
history_cols = ["card1_cnt_txn_hist","card1_avg_txn_amt_hist","card1_time_secs_since_last_txn_hist"]
history_cols

['card1_cnt_txn_hist',
 'card1_avg_txn_amt_hist',
 'card1_time_secs_since_last_txn_hist']

#### Model Results with History Columns

In [19]:
feature_cols_hist=feature_cols+history_cols

In [20]:
feature_cols_hist

['TransactionAmt',
 'ProductCD',
 'card1',
 'card2',
 'card3',
 'card4',
 'card5',
 'card6',
 'addr1',
 'addr2',
 'P_emaildomain',
 'R_emaildomain',
 'DeviceType',
 'DeviceInfo',
 'log_txnamt',
 'has_identity',
 'card1_cnt_txn_hist',
 'card1_avg_txn_amt_hist',
 'card1_time_secs_since_last_txn_hist']

In [21]:
hist_feature_result = model_results(train_h, valid_h, feature_cols_hist)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Number of positive: 16599, number of negative: 455833
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.024212 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2573
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 19
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


#### Comparing the results

In [22]:
df_comp=pd.DataFrame([
    {
        "version":"baseline",
        "n_feature":baseline_result["n_features"],
        "roc_auc":baseline_result["roc_auc"],
        "pr_auc":baseline_result["pr_auc"],
    },
    {
        "version":"hist_feature_result",
        "n_feature":hist_feature_result["n_features"],
        "roc_auc":hist_feature_result["roc_auc"],
        "pr_auc":hist_feature_result["pr_auc"],
    }
])

In [23]:
df_comp.sort_values(["pr_auc", "roc_auc"], ascending=False)

,version,n_feature,roc_auc,pr_auc
0,baseline,16,0.807761,0.230621
1,hist_feature_result,19,0.796004,0.207321


```sql
In this notebook we are able to achieve:
- history features for a particular attribute "card1"
- Ensured that history features don't leak and get computed only just before txn
- a comparison table showing whether it trually helped

```

#### End